# U-Net vs VAE: same encoder–decoder bones, different jobs

**Thesis:** Autoencoders/VAEs learn to **reconstruct** images; U-Nets learn to **segment** them. The architectures rhyme, but the **head, loss, and supervision** differ—so do the outcomes.

## Latent walks (what and why)
A **latent walk** is decoding a smooth path between two latent codes.
- Pick two codes \(z_0, z_1\) (e.g., encodings of two images).
- Interpolate \(z(t) = (1-t)z_0 + t z_1\) for \(t \in [0,1]\) (or use slerp).
- Decode each \(z(t)\) to an image.
**Why it matters:** Good VAEs map images to a **continuous manifold**, so small moves in \(z\) produce small, meaningful visual changes (pose, color, style). It’s a sanity check that the model learned a structured latent space rather than memorizing pixels.

## Why segmentation?
If we want the model to “know the pet,” we want a **per-pixel mask** of the subject, not the whole background. The mask is the **ground truth (GT)**: white = foreground, black = background.

## Why U-Net?
U-Net adds **skip connections**: it passes high-resolution encoder features directly to the decoder at the same scale (concatenate, then convolve). This recovers edges lost during downsampling, yielding **sharper boundaries** and higher overlap metrics (IoU/Dice).

## What we’ll demo (Oxford-IIIT Pet, binary masks)

### 1) Reconstruction track
- **Models:** `ConvAE`, `ConvVAE`
- **How:** train to minimize pixel reconstruction loss (BCE-with-logits). Show recon grids and a latent walk.
- **What you’ll learn (why this demo):**
  - VAEs produce **smoother, more coherent** reconstructions than AEs.
  - Latent space is **continuous**: interpolations change images smoothly.
  - Reconstruction ≠ understanding of object boundaries.


#### Auto-Encoder (AE)

Image x ──▶ [ Encoder f(ϕ) ] ──▶ [  z  ] ──▶ [ Decoder g(θ) ] ──▶ x_hat
              (downsample)         (latent)      (upsample)

(reconstruction loss: compare x_hat vs x)


#### Variational Auto-Encoder (VAE) - reconstruction + KL

Image x ──▶ [ Encoder f(ϕ) ] ──▶  μ(x)
                           └──▶  logσ²(x) ──▶ σ = exp(0.5·logσ²)

μ(x) ──▶
        └──▶ [ reparameterize: z = μ + σ·ε ] ──▶ [ Decoder g(θ) ] ──▶ x_hat
σ     ──▶

(reconstruction loss: x_hat vs x)   (+ KL(q(z|x) || p(z)=N(0,I)) regularization)


### 2) Segmentation track
- **Models:** `UNetSeg` vs `UNetNoSkipSeg` (same capacity; the only change is **skips**)
- **How:** train with overlap-aware loss (BCE-with-logits + Dice). Evaluate IoU at 0.5 and at the **best threshold** found on validation.
- **What you’ll learn (why this demo):**
  - **Skips matter:** U-Net predicts **crisper masks** and preserves thin p


#### U-Net with skip connections disabled
Image x ──▶ [ Encoder f(ϕ) ] ──▶  μ(x)
                           └──▶  logσ²(x) ──▶ σ = exp(0.5·logσ²)

μ(x) ──▶
        └──▶ [ reparameterize: z = μ + σ·ε ] ──▶ [ Decoder g(θ) ] ──▶ x_hat
σ     ──▶

(reconstruction loss: x_hat vs x)   (+ KL(q(z|x) || p(z)=N(0,I)) regularization)


#### U-Net (with skip connections) — segmentation

Image x ──▶ [ Encoder ] ──▶ [ Bottleneck ] ──▶ [ Decoder ] ──▶ [ Head 1×1 ] ──▶  Mask  (sigmoid)

**Down path**:  H → H/2 → H/4 </br>
**Bottleneck**: @ H/8 (no sampling) </br>
**Up path**:    H/4 → H/2 → H </br>
**Skips**: encoder features at H, H/2, H/4 → concat into decoder at matching scales </br>
**Loss**:  BCE-with-logits + Dice (Mask vs GT)


#### Include libraries, paths and parameters

In [ ]:
import os
from contextlib import contextmanager
import torch
import torch.nn as nn, torch.nn.functional as F
import torchvision as tv, torchvision.transforms as T
import numpy as np
from torch.utils.data import DataLoader, Subset
from PIL import Image

import matplotlib.pyplot as plt
import math

HOME_DIR = f"{os.environ['HOME']}/Documents/GitHub/Generative-AI"
project = "UNet_v_VAE"
dataset_path = f"{HOME_DIR}/datasets/{project}"
result_path = f"{HOME_DIR}/results/{project}"
model_path = f"{result_path}/models"
import sys
sys.path += [dataset_path, result_path, model_path]

# Determine the GPU type (cuda for Nvidia, mps for Mac, cpu if neither available)
from support import get_device
device = get_device()

# Parameters
imgSize = 256
epochs = 100
lr0 = 1e-3
min_lr = 1e-5
warmup_epochs = 5


# Depending on the GPU device, select the appropriate precision to ensure accuracy and reduce memory usage
@contextmanager
def smart_autocast(device: torch.device, enabled=True):
    if not enabled:
        yield
    elif device.type == "cuda":
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            yield
    elif device.type == "mps":
        # MPS AMP exists; BF16 is safest. If you see weirdness, set enabled=False.
        with torch.autocast(device_type="mps", dtype=torch.bfloat16):
            yield
    else:
        yield  # CPU: no autocast


### Helpers - for evaluation / understanding
These helpers compare the performance of various NN architectures through visualizations:
<ul><li>Convolutional Auto-Encoder</li>
<li>Convolutional Variational Auto-Encoder (AE with regularization)</li>
<li>U-Net: similar to a VAE with 'skip' operation from encoder to decoder side</li>
<li>U-Net "no-skip": a U-Net with the 'skip' disabled</li></ul>
They also compare effects of key parameters:<ul>
<li>Latent encodings on a VAE for segmentation quality</li>
<li>Selection threshold (0-1) effect on how well an image is segmented from the background (IoU)</li></ul>

In [ ]:
@torch.no_grad()
def save_unet_progress(unet_model, loader, device, path="unet_progress.png", thr=0.5, n=4):
    '''Display U-Net border performance. Shows input image, the ground truth mask (GT), the predicted border, and the error.
    In this case, it assumes a threshold cutoff at P=0.5, which may not be where peak performance lies. However, a well-behaved
    U-Net will at least be close to its peak at this point.'''
    unet_model.eval()
    x, y = next(iter(loader))
    x, y = x.to(device), y.to(device)
    p = torch.sigmoid(unet_model(x))
    m = (p > thr).float()
    err = (p - y).abs()  # probability error map

    x, y, m, err = x[:n].cpu(), y[:n].cpu(), m[:n].cpu(), err[:n].cpu()

    rows, cols = 4, n
    fig = plt.figure(figsize=(3*cols, 3*rows))
    for i in range(n):
        # row 1: input
        ax = fig.add_subplot(rows, cols, 1+i); ax.axis("off")
        img = x[i] if x.shape[1]==3 else x[i].repeat(3,1,1)
        ax.set_title("Input" if i==0 else "")
        ax.imshow(img.permute(1,2,0).numpy())
        # row 2: GT
        ax = fig.add_subplot(rows, cols, 1*cols+i+1); ax.axis("off")
        ax.set_title("GT" if i==0 else "")
        ax.imshow(y[i,0].numpy(), cmap="gray")
        # row 3: Pred
        ax = fig.add_subplot(rows, cols, 2*cols+i+1); ax.axis("off")
        ax.set_title(f"Pred@{thr:.2f}" if i==0 else "")
        ax.imshow(m[i,0].numpy(), cmap="gray")
        # row 4: |prob − GT|
        ax = fig.add_subplot(rows, cols, 3*cols+i+1); ax.axis("off")
        ax.set_title("|prob − GT|" if i==0 else "")
        ax.imshow(err[i,0].numpy(), cmap="gray", vmin=0, vmax=1)
    fig.tight_layout(); fig.savefig(path, dpi=160); plt.close(fig)

@torch.no_grad()
def save_noskip_vs_unet(noskip_model, unet_model, loader, device, path="noskip_vs_unet.png", thr=0.5, n=4):
    '''Architecture payoff of using U-Net vs the 'no-skip' of AE/VAE. It shows the input image, the ground truth mask,
    and the mask estimate from both the U-Net and the 'no-skip'. '''
    noskip_model.eval(); unet_model.eval()
    x, y = next(iter(loader))
    x, y = x.to(device), y.to(device)
    p0 = torch.sigmoid(noskip_model(x))
    p1 = torch.sigmoid(unet_model(x))
    m0 = (p0 > thr).float()
    m1 = (p1 > thr).float()
    x, y, m0, m1 = x[:n].cpu(), y[:n].cpu(), m0[:n].cpu(), m1[:n].cpu()

    rows, cols = n, 4
    fig = plt.figure(figsize=(3*cols, 3*rows))
    for i in range(n):
        ax = fig.add_subplot(rows, cols, 4*i+1); ax.axis("off")
        img = x[i] if x.shape[1]==3 else x[i].repeat(3,1,1)
        ax.set_title("Input" if i==0 else ""); ax.imshow(img.permute(1,2,0).numpy())
        ax = fig.add_subplot(rows, cols, 4*i+2); ax.axis("off")
        ax.set_title("GT" if i==0 else ""); ax.imshow(y[i,0].numpy(), cmap="gray")
        ax = fig.add_subplot(rows, cols, 4*i+3); ax.axis("off")
        ax.set_title("No-skip pred" if i==0 else ""); ax.imshow(m0[i,0].numpy(), cmap="gray")
        ax = fig.add_subplot(rows, cols, 4*i+4); ax.axis("off")
        ax.set_title("U-Net pred" if i==0 else ""); ax.imshow(m1[i,0].numpy(), cmap="gray")
    fig.tight_layout(); fig.savefig(path, dpi=160); plt.close(fig)

@torch.no_grad()
def save_unet_examples(unet_model, loader, device, path="noskip_vs_unet.png", thr=0.5, n=4):
    '''Shows several examples of the U-Net generating successful masks. It compares the input image with the computed mask.
    This is used to create fun visuals of well-performing cases. The 'skips' list removes cases which do not perform 
    especially well. '''
    # skip images I don't like
    skips = [4,6,8,11,12,16,20]
    skips.sort(reverse=True)
    x, y = next(iter(loader))
    x, y = x.to(device), y.to(device)
    p1 = torch.sigmoid(unet_model(x))
    m1 = (p1 > thr).float()
    x, y, m1 = x[:n+len(skips)].cpu(), y[:n+len(skips)].cpu(), m1[:n+len(skips)].cpu()
    def remove_indices(t, indices):
        indices.sort(reverse=True) # must go in descending so the lower-number index are accurate
        for ix in indices:
            t = torch.cat((t[:ix], t[ix + 1:]))
        return t
    x = remove_indices(x, skips)
    y = remove_indices(y, skips)
    m1 = remove_indices(m1, skips)
    rows, cols = n//4, 8
    fig = plt.figure(figsize=(3*cols, 3*rows))
    for i in range(rows):
        for j in range(cols//2):
            ax = fig.add_subplot(rows, cols, cols*i+2*j+1); ax.axis("off")
            img = x[i*cols//2 + j]
            ax.set_title("Input"); ax.imshow(img.permute(1,2,0).numpy())
            ax = fig.add_subplot(rows, cols, cols*i+2*j+2); ax.axis("off")
            ax.set_title("U-Net mask"); ax.imshow(m1[i*4+j,0].numpy(), cmap="gray")
    fig.tight_layout(); fig.savefig(path, dpi=160); plt.close(fig)


In [ ]:
@torch.no_grad()
def save_threshold_curve(unet_model, unet_noskip_model, loader, device, path="iou_vs_threshold.png"):
    '''Comparative KPI for U-Net vs the "no-skip" version. It computes the mask quality (IoU) across the 
    classification threshold for both architectures. A successfully performing U-Net should show better coverage.'''
    unet_model.eval()
    unet_noskip_model.eval()
    x_un, x_ns, ys = [], [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        prob_un = torch.sigmoid(unet_model(x))
        prob_ns = torch.sigmoid(unet_noskip_model(x))
        x_un.append(prob_un.cpu())
        x_ns.append(prob_ns.cpu()) 
        ys.append(y.cpu())
        break  # one batch is fine for the picture
    prob_un = torch.cat(x_un, 0); prob_ns = torch.cat(x_ns, 0)
    y = torch.cat(ys, 0)

    thrs = [i/100 for i in range(1,100)]
    scores_un = []; scores_ns = []
    for t in thrs:
        # U-net metrics
        pred_un = (prob_un > t).float()
        inter_un = (pred_un*y).sum((1,2,3))
        union_un = (pred_un + y - pred_un*y).sum((1,2,3)).clamp_min(1)
        scores_un.append((inter_un/union_un).mean().item())
        # no-skip metrics
        pred_ns = (prob_ns > t).float()
        inter_ns = (pred_ns*y).sum((1,2,3))
        union_ns = (pred_ns + y - pred_ns*y).sum((1,2,3)).clamp_min(1)
        scores_ns.append((inter_ns/union_ns).mean().item())

    import matplotlib.pyplot as plt
    plt.figure(figsize=(6,4))
    plt.plot(thrs, scores_un, color='b', label="U-Net")
    plt.plot(thrs, scores_ns, color='r', label="no-skip")
    plt.xlabel("Threshold"); plt.ylabel("IoU"); plt.title("IoU vs Threshold")
    plt.legend()
    plt.tight_layout(); plt.savefig(path, dpi=160); plt.close()


In [ ]:
@torch.no_grad()
def sample_pair(loader, device, i0=0, i1=1):
    """Get a consistent pair (x0,x1) from the first batch."""
    xb, _ = next(iter(loader))          # assumes loader is val (shuffle=False)
    xb = xb.to(device)
    return xb[i0:i0+1], xb[i1:i1+1]


@torch.no_grad()
def save_vae_recons(model, loader, device, path="vae_recon.png", n=4):
    model.eval()
    xb, _ = next(iter(loader))
    xb = xb.to(device)[:n]
    # run full forward if you prefer; but safer to call encode→reparam→decode explicitly
    mu, logvar = model.encode(xb)
    std = torch.exp(0.5 * logvar)
    eps = torch.randn_like(std)
    z = mu + std * eps
    xh = model.decode(z).clamp(0,1)

    import matplotlib.pyplot as plt
    def to_disp(img):
        img = img.detach().float().cpu()
        if img.size(0) == 1: img = img.repeat(3,1,1)
        else: img = img[:3]
        return img.permute(1,2,0).clamp(0,1).numpy()

    fig = plt.figure(figsize=(3*n, 6), dpi=120)
    for i in range(n):
        ax = fig.add_subplot(2, n, 1+i); ax.axis("off")
        if i == 0: ax.set_title("Input")
        ax.imshow(to_disp(xb[i]))
        ax = fig.add_subplot(2, n, n+1+i); ax.axis("off")
        if i == 0: ax.set_title("Recon")
        ax.imshow(to_disp(xh[i]))
    plt.tight_layout(); fig.savefig(path, bbox_inches="tight"); plt.close(fig)


@torch.no_grad()
def _latent_mean(model, x):
    """Return the mean latent for AE or VAE."""
    out = model.encode(x)
    if isinstance(out, (list, tuple)):
        # VAE style: (mu, logvar, ...) -> use mu
        z = out[0]
    elif isinstance(out, dict):
        z = out.get("mu", out.get("z"))
        if z is None:
            # fallback: first tensor value
            z = next(v for v in out.values() if isinstance(v, torch.Tensor))
    else:
        # AE style: encode returns z directly
        z = out
    return z

def _slerp(z0, z1, t, eps=1e-6):
    """
    SLERP for conv latents (B,C,H,W) or (B,D). We compute the angle on the
    flattened vectors, but apply the weights to the original tensors.
    """
    b = z0.shape[0]
    f0 = z0.view(b, -1); f1 = z1.view(b, -1)
    f0n = F.normalize(f0, dim=1); f1n = F.normalize(f1, dim=1)
    dot = (f0n * f1n).sum(1).clamp(-1+1e-6, 1-1e-6)                 # (B,)
    omega = torch.acos(dot)                                         # (B,)
    so = torch.sin(omega) + eps
    # broadcast weights back to latent shape
    w0 = (torch.sin((1-t)*omega) / so).view(b, *([1]*(z0.dim()-1)))
    w1 = (torch.sin(    t*omega) / so).view(b, *([1]*(z0.dim()-1)))
    return w0 * z0 + w1 * z1

def _lerp(z0, z1, t):
    return (1-t)*z0 + t*z1

@torch.no_grad()
def save_latent_walk(model, x0=None, x1=None, loader=None, device=None,
                             steps=8, use_slerp=True, path="latent_walk.png",
                             from_logits=False, title=None, show_originals=True):
    """
    Works for BOTH AE and VAE as long as model has encode()/decode().
    - If x0/x1 not given, samples first two from loader.
    - Interpolates in latent space between mean latents (mu for VAE, z for AE).
    - from_logits: set True if decode() returns logits (yours is False).
    """
    model.eval()
    if (x0 is None or x1 is None):
        assert loader is not None and device is not None
        xb,_ = next(iter(loader))
        x0, x1 = xb[:1].to(device), xb[1:2].to(device)

    z0 = _latent_mean(model, x0)
    z1 = _latent_mean(model, x1)

    ts = torch.linspace(0, 1, steps, device=z0.device)
    imgs = []
    for i, t in enumerate(ts):
        zt = _slerp(z0, z1, t) if use_slerp else _lerp(z0, z1, t)
        out = model.decode(zt)
        if from_logits: out = torch.sigmoid(out)
        imgs.append(out.clamp(0,1).squeeze(0).cpu())

    def disp(ax, img, ttl=None):
        ax.axis("off")
        if ttl: ax.set_title(ttl, fontsize=8)
        img3 = img[:3].permute(1,2,0).numpy()
        ax.imshow(img3)

    # figure: x0 | t=... | ... | x1
    cols = steps
    if show_originals:
        cols += 2 # include originals at start and end of series

    fig = plt.figure(figsize=(1.6*cols, 2.2), dpi=150)
    if title: fig.suptitle(title, y=1.02, fontsize=10)

    if show_originals:
        ax = fig.add_subplot(1, cols, 1); disp(ax, x0[0].cpu(), "x0")
        start = 2
    else:
        start = 1
    for k, im in enumerate(imgs, start=start):
        ax = fig.add_subplot(1, cols, k)
        disp(ax, im, f"t={ts[k-start]:.2f}")
    if show_originals:
        ax = fig.add_subplot(1, cols, cols); disp(ax, x1[0].cpu(), "x1")

    fig.savefig(path, bbox_inches="tight"); plt.close(fig)



In [ ]:

@torch.no_grad()
def save_summary_sheet(
    convae_model, convvae_model, unet_model, noskip_model,
    loader, device, path="summary_sheet.png", thr=0.5, n=4
):
    """
    Make a grid with columns:
      [Input | AE recon | VAE recon | GT | U-Net pred | No-skip pred]
    using the SAME n images from the first batch of `loader`.

    Assumptions:
      - Models output logits for seg (U-Net / No-skip), need sigmoid.
      - AE/VAE decoders output [0,1] (sigmoid) or raw in [0,1].
      - Images are floats in [0,1]. If you normalized, denorm before showing.
    """
    # switch to eval, remember state
    models = [convae_model, convvae_model, unet_model, noskip_model]
    was_training = [m.training for m in models]
    for m in models: m.eval()

    # fixed batch
    xb, yb = next(iter(loader))
    xb, yb = xb.to(device), yb.to(device)
    n = min(n, xb.size(0))

    # --- AE / VAE reconstructions ---
    # ConvAE: x -> x_hat
    ae_recon = convae_model(xb[:n]).clamp(0,1)

    # ConvVAE: x -> (x_hat, mu, logvar) or just x_hat depending on your forward
    vae_out = convvae_model(xb[:n])
    if isinstance(vae_out, (list, tuple)) and len(vae_out) >= 1:
        vae_recon = vae_out[0]
    else:
        vae_recon = vae_out
    vae_recon = vae_recon.clamp(0,1)

    # --- Segmentation preds (logits -> prob -> mask) ---
    unet_prob   = torch.sigmoid(unet_model(xb[:n]))
    noskip_prob = torch.sigmoid(noskip_model(xb[:n]))
    unet_pred   = (unet_prob   > thr).float()
    noskip_pred = (noskip_prob > thr).float()

    # --- helpers for visualization ---
    def to_disp(img):  # (C,H,W) -> (H,W,3) CPU numpy
        img = img.detach().float().cpu()
        if img.ndim == 3:
            if img.shape[0] == 1:
                img = img.repeat(3,1,1)
            elif img.shape[0] > 3:
                img = img[:3]  # just in case
            return img.permute(1,2,0).clamp(0,1).numpy()
        raise ValueError("Expected CHW image tensor")

    def to_mask(mask):  # (1,H,W) -> (H,W) grayscale
        m = mask.detach().float().cpu()
        if m.ndim == 3 and m.shape[0] == 1:
            m = m[0]
        return m.clamp(0,1).numpy()

    # --- build figure ---
    titles = ["Input", "AE recon", "VAE recon", "GT", "UNet pred", "No-skip pred"]
    cols = len(titles)
    rows = n
    h = max(2.2 * rows, 3.0)
    w = max(2.8 * cols, 8.0)
    fig = plt.figure(figsize=(w, h), dpi=150)

    for i in range(n):
        # column 1: input
        ax = fig.add_subplot(rows, cols, i*cols + 1); ax.axis("off")
        ax.imshow(to_disp(xb[i]))
        if i == 0: ax.set_title(titles[0], fontsize=10)

        # column 2: AE recon
        ax = fig.add_subplot(rows, cols, i*cols + 2); ax.axis("off")
        ax.imshow(to_disp(ae_recon[i]))
        if i == 0: ax.set_title(titles[1], fontsize=10)

        # column 3: VAE recon
        ax = fig.add_subplot(rows, cols, i*cols + 3); ax.axis("off")
        ax.imshow(to_disp(vae_recon[i]))
        if i == 0: ax.set_title(titles[2], fontsize=10)

        # column 4: GT mask
        ax = fig.add_subplot(rows, cols, i*cols + 4); ax.axis("off")
        ax.imshow(to_mask(yb[i]), cmap="gray")
        if i == 0: ax.set_title(titles[3], fontsize=10)

        # column 5: UNet pred
        ax = fig.add_subplot(rows, cols, i*cols + 5); ax.axis("off")
        ax.imshow(to_mask(unet_pred[i]), cmap="gray")
        if i == 0: ax.set_title(titles[4] + f" @ {thr:.2f}", fontsize=10)

        # column 6: No-skip pred
        ax = fig.add_subplot(rows, cols, i*cols + 6); ax.axis("off")
        ax.imshow(to_mask(noskip_pred[i]), cmap="gray")
        if i == 0: ax.set_title(titles[5] + f" @ {thr:.2f}", fontsize=10)

    plt.tight_layout()
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)

    # restore train/eval states
    for m, was in zip(models, was_training):
        if was: m.train()



### Model Building blocks
We want to demonstrate this using the same basic blocks for either architecture

| Block          | U-Net (seg)            | ConvVAE (recon)                |
| -------------- | ---------------------- | ------------------------------ |
| DoubleConv     | ✅ encoder + decoder    | ✅ encoder + decoder            |
| Down           | ✅                      | ✅                              |
| Up             | ✅ + **concat skip**    | ✅ **no skip**                  |
| OutHead        | ✅ → `n_classes` logits | ✅ → `in_ch` logits             |
| μ / logσ       | —                      | ✅ 1×1 conv heads               |
| latent_to_feat | —                      | ✅ 1×1 conv to decoder channels |


In [ ]:

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch, groups=8):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(groups, out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(groups, out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x): 
        return self.block(x)


# Define decoders for skip-concat or no skip-concat
class UpNoSkip(nn.Module):
    """Decoder step without skip-concat (for AE/VAE or no-skip seg)."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.conv = DoubleConv(in_ch, out_ch)
    def forward(self, x):
        return self.conv(self.up(x))
    
class UpUNet(nn.Module):
    """Decoder step with skip-concat (for U-Net)."""
    def __init__(self, up_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.conv = DoubleConv(up_ch + skip_ch, out_ch)  # CONCAT
    def forward(self, x, skip):
        x = self.up(x)
        # pad/crop to align in case of off-by-1
        dh, dw = skip.size(2)-x.size(2), skip.size(3)-x.size(3)
        if dh or dw:
            x = F.pad(x, (0,max(0,dw),0,max(0,dh)))
            x = x[:, :, :skip.size(2), :skip.size(3)]
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)



# --------- Down block: maxpool then DoubleConv ----------
class Down(nn.Module):
    def __init__(self, c_in, c_out):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.conv = DoubleConv(c_in, c_out)
    def forward(self, x):
        return self.conv(self.pool(x))
    

# Define heads for case when used for reconstruction (ReconHead) or segmentation (SegHead)
class ReconHead(nn.Module):
    """For AE/VAE reconstruction: 3-channel image in [0,1]."""
    def __init__(self, in_ch, out_ch=3):
        super().__init__()
        self.out = nn.Conv2d(in_ch, out_ch, kernel_size=1)
    def forward(self, x):
        return torch.sigmoid(self.out(x))  # if inputs are in [0,1]

class SegHead(nn.Module):
    """For segmentation: 1-channel logits (use BCEWithLogits + Dice)."""
    def __init__(self, in_ch, n_classes=1):
        super().__init__()
        self.out = nn.Conv2d(in_ch, n_classes, kernel_size=1)
    def forward(self, x):
        return self.out(x)  # logits


### Models

In [ ]:

class UNetSeg(nn.Module):
    def __init__(self, in_ch=3, n_classes=1, base=32):
        super().__init__()
        self.enc1 = DoubleConv(in_ch, base)          # 32
        self.enc2 = Down(base, base*2)               # 64
        self.enc3 = Down(base*2, base*4)             # 128
        self.enc4 = Down(base*4, base*8)             # 256
        self.bottleneck = Down(base*8, base*16)      # 512

        self.up4 = UpUNet(up_ch=base*16, skip_ch=base*8,  out_ch=base*8)   # 512↑ + 256 → 256
        self.up3 = UpUNet(up_ch=base*8,  skip_ch=base*4,  out_ch=base*4)   # 256↑ + 128 → 128
        self.up2 = UpUNet(up_ch=base*4,  skip_ch=base*2,  out_ch=base*2)   # 128↑ +  64 →  64
        self.up1 = UpUNet(up_ch=base*2,  skip_ch=base,    out_ch=base)     #  64↑ +  32 →  32

        self.head = SegHead(base, n_classes)  # in UNetSeg

    def forward(self, x):
        e1 = self.enc1(x); e2 = self.enc2(e1); e3 = self.enc3(e2); e4 = self.enc4(e3)
        b  = self.bottleneck(e4)
        d4 = self.up4(b, e4); d3 = self.up3(d4, e3); d2 = self.up2(d3, e2); d1 = self.up1(d2, e1)
        return self.head(d1)


In [ ]:

class UNetNoSkipSeg(nn.Module):
    def __init__(self, in_ch=3, n_classes=1, base=32):
        super().__init__()
        self.enc1 = DoubleConv(in_ch, base)
        self.enc2 = Down(base, base*2)
        self.enc3 = Down(base*2, base*4)
        self.enc4 = Down(base*4, base*8)
        self.bottleneck = Down(base*8, base*16)

        self.up4 = UpNoSkip(base*16, base*8)
        self.up3 = UpNoSkip(base*8, base*4)
        self.up2 = UpNoSkip(base*4, base*2)
        self.up1 = UpNoSkip(base*2, base)

        self.head = SegHead(base, n_classes)

    def forward(self, x):
        e1 = self.enc1(x); e2 = self.enc2(e1); e3 = self.enc3(e2); e4 = self.enc4(e3)
        b  = self.bottleneck(e4)
        d4 = self.up4(b); d3 = self.up3(d4); d2 = self.up2(d3); d1 = self.up1(d2)
        return self.head(d1)  # logits


#### Convolutional Auto-Encoder (Conv AE)

In [ ]:
class ConvAE(nn.Module):
    def __init__(self, in_ch=3, base=32):
        super().__init__()
        self.e1 = DoubleConv(in_ch, base)
        self.e2 = Down(base, base*2)
        self.e3 = Down(base*2, base*4)
        self.b  = Down(base*4, base*8)

        self.u3 = UpNoSkip(base*8, base*4)
        self.u2 = UpNoSkip(base*4, base*2)
        self.u1 = UpNoSkip(base*2, base)

        self.head = ReconHead(base, out_ch=in_ch)
    # encode and decode needed for demos. Allows for latent-space distortion.
    def encode(self, x):
        h = self.e1(x); h = self.e2(h); h = self.e3(h); h = self.b(h)
        return h  # latent feature map (B, base*8, H/8, W/8)

    def decode(self, z):
        h = self.u3(z); h = self.u2(h); h = self.u1(h)
        return self.head(h)  # (B,3,H,W) in [0,1]

    def forward(self, x):
        return self.decode(self.encode(x))


#### Convolutional Variational Auto-Encoder (Conv VAE)

In [ ]:
class ConvVAE(nn.Module):
    def __init__(self, in_ch=3, z_dim=128, base=32):
        super().__init__()
        # encoder
        self.e1 = DoubleConv(in_ch, base)
        self.e2 = Down(base, base*2)
        self.e3 = Down(base*2, base*4)
        self.b  = Down(base*4, base*8)
        # map to latent
        self.to_mu     = nn.Conv2d(base*8, z_dim, kernel_size=1)
        self.to_logvar = nn.Conv2d(base*8, z_dim, kernel_size=1)
        # decoder (mirror)
        self.d_in = nn.Conv2d(z_dim, base*8, kernel_size=1)
        self.u3 = UpNoSkip(base*8, base*4)
        self.u2 = UpNoSkip(base*4, base*2)
        self.u1 = UpNoSkip(base*2, base)
        self.head = ReconHead(base, out_ch=in_ch)

    def reparam(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + std * eps

    def encode(self, x):
        h = self.e1(x); h = self.e2(h); h = self.e3(h); h = self.b(h)
        return self.to_mu(h), self.to_logvar(h)

    def decode(self, z):
        h = self.d_in(z)
        h = self.u3(h); h = self.u2(h); h = self.u1(h)
        return self.head(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparam(mu, logvar)
        x_hat = self.decode(z)
        return x_hat, mu, logvar


### Load data

#### `OxfordPetsSeg`: what it does
- Thin wrapper around **Oxford-IIIT Pet** that returns images and **binary masks** so we can train a fast U-Net demo.
  - Downloads (if needed), resizes to a fixed size (e.g., 256×256), converts masks to `{0,1}`.
  - Keeps `H` and `W` divisible by 8 so down/upsampling lines up.
- `bin_rule` controls **how original mask IDs become binary**:
  - `non_background`: foreground = any pixel **not** equal to the dominant background ID (simple; may include “border” pixels).
  - `pet_eq_max`: foreground = pixels with the **max** class ID (works on some torchvision variants).
  - `mask_gt_1`: foreground = pixels with ID **> 1** (merges pet + border as one foreground).
- For the cleanest demo: **pet-only** → `foreground = (seg == 1)` (exact ID can vary by torchvision build).


In [ ]:
# In your OxfordPetsSeg class:

class OxfordPetsSeg(torch.utils.data.Dataset):
    def __init__(self, root="./datasets", split="trainval", size=256, rgb=True, bin_rule="non_background", pet_id=1):
        self.ds = tv.datasets.OxfordIIITPet(
            root=root, download=True, target_types=("segmentation",), split=split
        )
        self.size, self.rgb, self.bin_rule = size, rgb, bin_rule
        self.pet_id = pet_id  # <-- NEW: lets you override the pet class id

        self.t_img = T.Compose([T.Resize((size,size)), T.ToTensor()])
        self.t_msk = T.Resize((size,size), interpolation=T.InterpolationMode.NEAREST)

        # Peek a mask to guess background id (mode)
        _, m0 = self.ds[0]
        arr = np.array(m0, dtype=np.int64)
        vals, counts = np.unique(arr, return_counts=True)
        self.background_id = int(vals[np.argmax(counts)])
        print(f"[Pets] ids sample: {vals.tolist()} | bg≈{self.background_id} | bin_rule={self.bin_rule}")

    def to_binary(self, arr: np.ndarray) -> np.ndarray:
        if self.bin_rule == "non_background":
            return (arr != self.background_id).astype(np.float32)
        if self.bin_rule == "pet_eq_max":
            return (arr == arr.max()).astype(np.float32)
        if self.bin_rule == "mask_gt_1":
            return (arr > 1).astype(np.float32)
        # --- NEW: pet-only with configurable id (default 1) ---
        if self.bin_rule == "pet_eq_one":
            return (arr == self.pet_id).astype(np.float32)
        raise ValueError(f"Unknown bin_rule: {self.bin_rule}")

    def __len__(self): return len(self.ds)

    def __getitem__(self, i):
        img, seg = self.ds[i]
        x = self.t_img(img.convert("RGB") if self.rgb else img.convert("L"))
        seg = self.t_msk(seg)
        y = torch.from_numpy(self.to_binary(np.array(seg, dtype=np.int64))).unsqueeze(0)  # [1,H,W], {0,1}
        return x, y


In [ ]:
# 1) Datasets with consistent pet-only masks and augmentation for train
trainLen = 512
valLen = 64

train_tf_img = T.Compose([
    T.Resize((imgSize,imgSize)),
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(0.2, 0.2, 0.2, 0.05),
    T.RandomAffine(degrees=10, translate=(0.05,0.05), scale=(0.95,1.05), shear=5),
    T.ToTensor(),
])
val_tf_img = T.Compose([T.Resize((imgSize,imgSize)), T.ToTensor()])
tf_msk = T.Resize((imgSize,imgSize), interpolation=T.InterpolationMode.NEAREST)

# In OxfordPetsSeg __init__, pass transforms or add flags to switch
# Ensure to_binary does: return (arr == 1).astype(np.float32)

full = OxfordPetsSeg(size=imgSize, rgb=True, bin_rule="pet_eq_one", pet_id=1)
idx  = torch.randperm(len(full))[:trainLen+valLen]  
train_ds = torch.utils.data.Subset(full, idx[:trainLen].tolist())
val_ds   = torch.utils.data.Subset(full, idx[trainLen:].tolist())

# If class doesn’t accept external transforms, fork a tiny “OxfordPetsSegTrain/Val”
# that uses train_tf_img vs val_tf_img for images, and tf_msk for masks in both.

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False, num_workers=0)


### Instantiate Models and define optimizer functions

In [ ]:
# Instantiate & push to device (float32 is safest on MPS)
# The number of channels depends on the type of training images (e.g. RGB has 3 channels)
x0, _ = next(iter(train_loader))
IN_CH = x0.shape[1]


def pick_base(img):
    # simple, sane mapping
    if img <= 256:  return 32
    if img <= 384:  return 48
    return 64       # 512 or bigger

BASE = pick_base(imgSize)


# Instantiate the model
unet_model  = UNetSeg(in_ch=IN_CH, n_classes=1, base=BASE).to(device)
unet_noskip_model= UNetNoSkipSeg(in_ch=IN_CH, base=BASE).to(device)
conv_ae_model     = ConvAE(in_ch=IN_CH, base=BASE).to(device)
conv_vae_model    = ConvVAE(in_ch=IN_CH, z_dim=128, base=BASE).to(device)

# Add foreground prior bias
p = 0.2  # rough foreground prior
b = -math.log((1-p)/p)
with torch.no_grad():
    for m in (unet_model.head, unet_noskip_model.head):
        if hasattr(m, "out"): m.out.bias.fill_(b)   # if SegHead
        else: m.bias.fill_(b)                       # if plain Conv2d



### Loss Functions: we use IoU, and a combination of Dice loss and BCE-with-logits
---

#### IoU (Intersection over Union)
- Measures **overlap** between predicted mask `P` and ground truth `G`.
- Binary formula:
$$
  \mathrm{IoU} \;=\; \frac{|P \cap G|}{|P \cup G|} \;=\; \frac{\text{TP}}{\text{TP} + \text{FP} + \text{FN}}
$$
- Why use it for segmentation: focuses on **shape overlap** instead of being dominated by easy background pixels.

---

#### Dice loss vs. BCE-with-logits (and why combine them)
- **Dice coefficient** (overlap metric):
$$
  \mathrm{Dice}(p,y) \;=\; \frac{2\sum p\,y + \varepsilon}{\sum p + \sum y + \varepsilon}
  \quad\Rightarrow\quad
  \mathcal{L}_{\text{Dice}} = 1 - \mathrm{Dice}
  $$

  - Great when **foreground is small**; directly optimizes overlap.

- **BCE-with-logits** (`nn.BCEWithLogitsLoss`)
  - Binary cross-entropy computed on **logits** (numerically stable).
  - Trains **calibrated probabilities** at each pixel.

- **Why the combo (BCE + Dice)**
  - BCE makes probabilities honest; Dice pushes **shape overlap** to improve despite class imbalance.
  - In practice, `BCE + Dice` converges faster and yields **sharper masks** than either alone on small, imbalanced sets.


In [ ]:
# Logit heads → use BCEWithLogitsLoss (numerically stable)
bce_logits = nn.BCEWithLogitsLoss()


# ConvVAE KL on spatial latents (average so it's scale-invariant)
def kld_loss(mu, logvar):
    # mu/logvar: [B, z_dim, H/8, W/8] (or whatever your downsample factor is)
    return (-0.5 * (1 + logvar - mu.pow(2) - logvar.exp())).mean()


def dice_loss(prob, target, eps=1e-6):
    num = 2 * (prob*target).sum((1,2,3))
    den = (prob+target).sum((1,2,3)) + eps
    return (1 - (num+eps)/den).mean()

def unet_loss(logits, target, w_bce=1.0, w_dice=1.5):
    return w_bce*bce_logits(logits, target) + w_dice*dice_loss(torch.sigmoid(logits), target)


# IoU for logging (binary); use thresholds at 0.5 for demo
def iou_score(prob, target, thr=0.5, eps=1e-7):
    """
    prob: (N,1,H,W) in [0,1]; target: (N,1,H,W) binary {0,1}
    """
    pred = (prob > thr).float()
    inter = (pred * target).sum(dim=(1,2,3))
    union = (pred + target - pred * target).sum(dim=(1,2,3))
    iou = (inter + eps) / (union + eps)
    return iou.mean()

@torch.no_grad()
def best_iou(prob, y):
    best, best_t = 0.0, 0.5
    for t in [i/20 for i in range(1,20)]:  # 0.05..0.95
        pred = (prob > t).float()
        inter = (pred * y).sum((1,2,3))
        union = (pred + y - pred*y).sum((1,2,3)).clamp_min(1)
        iou = (inter/union).mean().item()
        if iou > best:
            best, best_t = iou, t
    return best, best_t


### Define optimizers and schedulers for AE, VAE, U-Net & No-skip

In [ ]:

# Define optimizers
unet_opt = torch.optim.AdamW(unet_model.parameters(), lr=lr0, weight_decay=1e-4)

unet_noskip_opt  = torch.optim.AdamW(unet_noskip_model.parameters(), lr=lr0, weight_decay=1e-4)
convae_opt  = torch.optim.AdamW(conv_ae_model.parameters(),  lr=lr0)
convvae_opt = torch.optim.AdamW(conv_vae_model.parameters(), lr=lr0)


# Warmup (LambdaLR) then Cosine; stitch them with SequentialLR
warmup = lambda e: max(1e-8, (e+1)/warmup_epochs)  # linear 0→1 over warmup_epochs
def make_sched(opt):
    return torch.optim.lr_scheduler.SequentialLR(
        opt,
        [torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda=warmup),
         torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs-warmup_epochs, eta_min=min_lr)],
        milestones=[warmup_epochs]
    )
sched_un  = make_sched(unet_opt)
sched_ns  = make_sched(unet_noskip_opt)
sched_ae  = make_sched(convae_opt)
sched_vae = make_sched(convvae_opt)


### Train step for AE, VAE, UNet, UNet NoSkip

In [ ]:
def wrap_step(name, sched):
    old = sched.step
    def _step(*args, **kwargs):
        if args or kwargs:
            print(f"[WARN] {name}.step called with args={args} kwargs={kwargs}")
        return old(*args, **kwargs)
    sched.step = _step

wrap_step("sched_un",  sched_un)
wrap_step("sched_ns",  sched_ns)
wrap_step("sched_ae",  sched_ae)
wrap_step("sched_vae", sched_vae)


In [ ]:
def recon_loss(x_hat, x):
    # better color than pure MSE
    return 0.8*torch.nn.functional.l1_loss(x_hat, x) + 0.2*torch.nn.functional.mse_loss(x_hat, x)


def train_one_epoch_convae(loader):
    conv_ae_model.train()
    total, total_recon, n = 0.0, 0.0, 0
    for x, _ in loader:
        x = x.to(device)
        convae_opt.zero_grad(set_to_none=True)
        x_hat = conv_ae_model(x)                 # returns reconstruction
        recon = recon_loss(x_hat, x)
        #recon = nn.functional.mse_loss(x_hat, x, reduction="mean")  # or BCE, your choice
        loss = recon                            # AE loss = recon only
        loss.backward(); convae_opt.step()

        bs = x.size(0)
        total += loss.item() * bs
        total_recon += recon.item() * bs
        n += bs
    avg = 1.0 / n
    return {"loss": total*avg, "recon": total_recon*avg, "kld": 0.0}


def train_one_epoch_convvae(loader, beta=1.0):
    conv_vae_model.train()
    total, total_recon, total_kld, n = 0.0, 0.0, 0.0, 0
    for x, _ in loader:
        x = x.to(device)
        convvae_opt.zero_grad(set_to_none=True)
        x_hat, mu, logvar = conv_vae_model(x)
        recon = recon_loss(x_hat, x)
        #recon = nn.functional.mse_loss(x_hat, x, reduction="mean")   # or BCE
        kld = (-0.5 * (1 + logvar - mu.pow(2) - logvar.exp())).mean()
        loss = recon + beta * kld
        loss.backward(); convvae_opt.step()

        bs = x.size(0); n += bs
        total += loss.item() * bs
        total_recon += recon.item() * bs
        total_kld += kld.item() * bs
    avg = 1.0 / n
    return {"loss": total*avg, "recon": total_recon*avg, "kld": total_kld*avg}


def train_one_epoch_seg(model, loader, optimizer, device, loss_fn):
    model.train()
    total_loss, total_iou, n = 0.0, 0.0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss   = loss_fn(logits, y)
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            prob = torch.sigmoid(logits)
            iou  = iou_score(prob, y, thr=0.5)
            bs   = x.size(0)
            total_loss += loss.item() * bs
            total_iou  += float(iou) * bs
            n += bs
    return {"loss": total_loss/max(n,1), "iou": total_iou/max(n,1)}



#### Evaluate for AE, VAE, UNet, UNet NoSkip

In [ ]:

@torch.no_grad()
def eval_convae(loader):
    conv_ae_model.eval()
    total, total_recon, n = 0.0, 0.0, 0
    for x, _ in loader:
        x = x.to(device)
        x_hat = conv_ae_model(x)
        #recon = nn.functional.mse_loss(x_hat, x, reduction="mean")
        recon = recon_loss(x_hat, x)
        loss = recon
        bs = x.size(0)
        total += loss.item() * bs
        total_recon += recon.item() * bs
        n += bs
    avg = 1.0 / n
    return {"loss": total*avg, "recon": total_recon*avg, "kld": 0.0}


@torch.no_grad()
def eval_convvae(loader, beta=1.0):
    conv_vae_model.eval()
    total, total_recon, total_kld, n = 0.0, 0.0, 0.0, 0
    for x, _ in loader:
        x = x.to(device)
        x_hat, mu, logvar = conv_vae_model(x)
        #recon = nn.functional.mse_loss(x_hat, x, reduction="mean")
        recon = recon_loss(x_hat, x)
        kld = (-0.5 * (1 + logvar - mu.pow(2) - logvar.exp())).mean()
        loss = recon + beta * kld
        bs = x.size(0); n += bs
        total += loss.item() * bs
        total_recon += recon.item() * bs
        total_kld += kld.item() * bs
    avg = 1.0 / n
    return {"loss": total*avg, "recon": total_recon*avg, "kld": total_kld*avg}


@torch.no_grad()
def eval_segmentation(model, loader, device, loss_fn, thresholds=(0.5,), return_curve=False):
    """
    Unified eval for any segmentation model that outputs logits.
    Returns: {
        "loss": float,
        "iou@0.5": float,        # always present
        "best_iou": float,       # max IoU across thresholds
        "best_thr": float,       # threshold that achieved best_iou
        # optional:
        "curve": List[(thr, iou)]
    }
    """
    was_training = model.training
    model.eval()

    total_loss = 0.0
    total_iou_per_thr = {thr: 0.0 for thr in thresholds}
    n = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = loss_fn(logits, y)
        probs = torch.sigmoid(logits)

        bs = x.size(0)
        total_loss += loss.item() * bs
        n += bs

        # accumulate IoU for each threshold
        for thr in thresholds:
            iou = iou_score(probs, y, thr=thr) 
            total_iou_per_thr[thr] += float(iou) * bs

    # restore training state
    if was_training:
        model.train()

    avg_loss = total_loss / max(n, 1)
    avg_iou_per_thr = {thr: (val / max(n, 1)) for thr, val in total_iou_per_thr.items()}

    # canonical keys
    iou_at_05 = avg_iou_per_thr.get(0.5, next(iter(avg_iou_per_thr.values()), 0.0))
    best_thr, best_iou = max(avg_iou_per_thr.items(), key=lambda kv: kv[1])

    out = {
        "loss": avg_loss,
        "iou@0.5": iou_at_05,
        "best_iou": best_iou,
        "best_thr": float(best_thr),
    }
    if return_curve:
        out["curve"] = sorted(avg_iou_per_thr.items())
    return out


def eval_unet(loader):
    return eval_segmentation(unet_model, loader, device, unet_loss, thresholds=(0.3, 0.4, 0.5, 0.6, 0.7))

def eval_unet_noskip(loader):
    return eval_segmentation(unet_noskip_model, loader, device, unet_loss, thresholds=(0.3, 0.4, 0.5, 0.6, 0.7))


### Train

In [ ]:
# parameters
num_epochs = 100
beta = 1.0  # β-VAE (set <1.0 to emphasize recon, >1.0 to emphasize disentangling)


def current_lr(optim):
    for g in optim.param_groups:
        return g["lr"]

# Save checkpoint -- to hold best models
def save_ckpt(path, model, epoch, metrics: dict):
    torch.save({
        "epoch": epoch,
        "state_dict": model.state_dict(),
        "metrics": metrics,
    }, path)
# Load checkpoint 
def load_ckpt(path, model, device):
    ck = torch.load(path, map_location=device)
    model.load_state_dict(ck["state_dict"])
    model.eval()
    return ck.get("epoch", None), ck.get("metrics", {})



#### AE / VAE

In [ ]:

best_ae = float("inf")
best_vae = float("inf")
patience = 20
wait_ae = 0
wait_vae = 0

ae_stop = False
vae_stop = False
for epoch in range(1, num_epochs + 1):
    if ae_stop == False:
        tr_ae  = train_one_epoch_convae(train_loader)                   # expects {'loss': ...}
        va_ae  = eval_convae(val_loader)                                # expects {'loss': ...}
        # step schedulers AFTER the epoch's optimizer steps
        sched_ae.step()
        # for AE/VAE use lowest val recon (or total loss) as “best”
        if va_ae['recon'] + 1e-6 < best_ae:
            best_ae, wait_ae = va_ae['recon'], 0
            best_ep_ae = epoch
            save_ckpt(f"{model_path}/conv_ae_best.pt", conv_ae_model, best_ep_ae, va_ae)
        else:
            wait_ae += 1
        if wait_ae >= patience:
            ae_stop = True

    if vae_stop == False:
        beta = min(1.0, epoch/10.0) # beta annealing -- ramp up beta over first 10 epochs
        tr_vae = train_one_epoch_convvae(train_loader, beta=beta)       # expects {'loss','recon','kld'}
        va_vae = eval_convvae(val_loader,     beta=beta)                # expects {'loss','recon','kld'}
        sched_vae.step()

        if va_vae['recon'] + 1e-6 < best_vae:
            best_vae, wait_vae = va_vae['recon'], 0
            best_ep_vae = epoch
            save_ckpt(f"{model_path}/conv_vae_best.pt", conv_vae_model, best_ep_vae, va_vae)
        else:
            wait_vae += 1
        if wait_vae >= patience:
            vae_stop = True


    if epoch % 10 == 0 or epoch == 1:
        print(f"[ConvAE] ep {epoch:02d} | train {tr_ae['loss']:.4f} "
              f"| recon {tr_ae['recon']:.4f} | kld {tr_ae['kld']:.4f} "
              f"|| val {va_ae['loss']:.4f} (recon {va_ae['recon']:.4f}, kld {va_ae['kld']:.4f})")
        print(f"[ConvVAE] ep {epoch:02d} | train {tr_vae['loss']:.4f} "
              f"| recon {tr_vae['recon']:.4f} | kld {tr_vae['kld']:.4f} "
              f"|| val {va_vae['loss']:.4f} (recon {va_vae['recon']:.4f}, kld {va_vae['kld']:.4f})")




#### U-Net / No-skip U-Net

In [ ]:
# distinct opts/schedulers
best_un, best_ns = -1.0, -1.0
best_ep_un = best_ep_ns = -1

for epoch in range(1, num_epochs+1):
    tr_un = train_one_epoch_seg(unet_model,        train_loader, unet_opt,   device, unet_loss)
    tr_ns = train_one_epoch_seg(unet_noskip_model, train_loader, unet_noskip_opt, device, unet_loss)

    va_un = eval_segmentation(unet_model,        val_loader, device, unet_loss,
                              thresholds=(0.3,0.4,0.5,0.6,0.7))
    va_ns = eval_segmentation(unet_noskip_model, val_loader, device, unet_loss,
                              thresholds=(0.3,0.4,0.5,0.6,0.7))

    # step schedulers *after* training
    sched_un.step()
    sched_ns.step()

    score_un = va_un["best_iou"]
    if score_un > best_un:
        best_un, best_ep_un = score_un, epoch
        save_ckpt(f"{model_path}/unet_best.pt", unet_model, best_ep_un, va_un)

    score_ns = va_ns["best_iou"]
    if score_ns > best_ns:
        best_ns, best_ep_ns = score_ns, epoch
        save_ckpt(f"{model_path}/unet_noskip_best.pt", unet_noskip_model, best_ep_ns, va_ns)

    if epoch == 1 or epoch % 10 == 0:
        print(f"[UNet]   ep {epoch:03d} | lr {unet_opt.param_groups[0]['lr']:.6f} "
              f"| tr IoU {tr_un['iou']:.3f} | val IoU@0.5 {va_un['iou@0.5']:.3f} "
              f"| best {va_un['best_iou']:.3f} @ thr={va_un['best_thr']:.2f}")
        print(f"[NoSkip] ep {epoch:03d} | lr {unet_noskip_opt.param_groups[0]['lr']:.6f} "
              f"| tr IoU {tr_ns['iou']:.3f} | val IoU@0.5 {va_ns['iou@0.5']:.3f} "
              f"| best {va_ns['best_iou']:.3f} @ thr={va_ns['best_thr']:.2f}")

print(f"Best UNet     val IoU {best_un:.3f} @ epoch {best_ep_un}")
print(f"Best No-Skip  val IoU {best_ns:.3f} @ epoch {best_ep_ns}")


### Results

In [ ]:

# load all models’ best checkpoints
_ = load_ckpt(f"{model_path}/conv_ae_best.pt",      conv_ae_model,       device)
_ = load_ckpt(f"{model_path}/conv_vae_best.pt",     conv_vae_model,     device)
_ = load_ckpt(f"{model_path}/unet_best.pt",        unet_model,         device)
_ = load_ckpt(f"{model_path}/unet_noskip_best.pt", unet_noskip_model,  device)


# U-Net visuals (same fixed batch inside helpers)
save_unet_progress(unet_model, val_loader, device, path=result_path + "/unet_progress_final.png")
save_threshold_curve(unet_model, unet_noskip_model, val_loader, device, path=result_path + "/iou_vs_threshold_final.png")
save_noskip_vs_unet(unet_noskip_model, unet_model, val_loader, device, thr=0.50,
                    path=result_path + "/noskip_vs_unet.png")

# Final 2x3 summary sheet (AE/VAE/UNet on the *same* images)
save_summary_sheet( conv_ae_model, conv_vae_model, unet_model, unet_noskip_model, val_loader, device,
    path=result_path + "/summary_sheet.png", thr=0.50, n=4
)


# visuals after AE/VAE training (uses a *fixed batch* inside helpers)
save_vae_recons(conv_vae_model, train_loader, device, path=result_path + "/vae_recon.png")

# Grab 2 fixed images to compare latent walk transition from ae & vae
# pick once from the *val* loader (shuffle=False)
x0, x1 = sample_pair(val_loader, device, i0=0, i1=1)

save_latent_walk(conv_vae_model, x0=x0, x1=x1,
                         steps=8, use_slerp=True, path=result_path + "/vae_latent_walk.png",
                         from_logits=False, title="VAE Latent Walk (decode(mu))")

save_latent_walk(conv_ae_model, x0=x0, x1=x1, steps=8, use_slerp=False,
        path=result_path + "/ae_latent_walk.png", from_logits=False, title="AE Latent Walk")




In [ ]:

save_unet_examples(unet_model, val_loader, device, path=result_path + "/unet_examples.png", thr=0.4, n=32)

In [ ]:


display(Image(filename=result_path + "/unet_examples.png"))

In [ ]:
num_pairs = 20
val_loader   = DataLoader(val_ds,   batch_size=num_pairs*2, shuffle=False, num_workers=0)
xb, _ = next(iter(val_loader))          # assumes loader is val (shuffle=False)
xb = xb.to(device)

for i in range(num_pairs):
    print(i)
    filename = result_path + f"/vae_latent_walk_{i}.png"
    save_latent_walk(conv_vae_model, x0=xb[2*i:2*i+1], x1=xb[2*i+1:2*i+2],
                         steps=8, use_slerp=True, path=filename,
                         from_logits=False, title="VAE Latent Walk (decode(mu))", show_originals=False)
    display(Image(filename=filename))


### Results - Image segmentation
We're train our models to segment the image into two classes: "subject" and "background". Ideally 

In [ ]:
from IPython.display import display, Image
takeaways = {"vae_recon.png": "Input image and estimate of the subject's border.",
             "vae_latent_walk.png": "A blend of 2 images in latent space. If there's regularization (VAE) this should be smooth.",
             "ae_latent_walk.png": "A blend of 2 images in latent space. If there's regularization (VAE) this should be smooth.",
             "unet_progress_final.png": "The input image, the ground truth border, predicted border at 0.5, pixel error map (black good, white miss)",
             "iou_vs_threshold_final.png": "Shows level of overlap between predicted and actual as function of cutoff threshold",
             "summary_sheet.png": "Input image, AE/VAE reconstruction, Ground Truth (GT), and the UNet model predicted border at 0.5",
             "noskip_vs_unet.png": "Comparison between regular UNet and the no-skip Unet"}
for img,note in takeaways.items():
    p = result_path + "/" + img
    print("Filename: ",p)
    print(note)
    display(Image(filename=p))


In [ ]:
def extract_recon(out, from_logits=False):
    """
    Return a BCHW tensor reconstruction from various model outputs:
    - Tensor -> returned directly
    - Tuple/List -> first BCHW tensor is used (e.g., (x_hat, mu, logvar))
    - Dict -> tries 'x_hat'/'recon'/'out' keys or first BCHW tensor value
    Optionally applies sigmoid if outputs are logits.
    """
    if isinstance(out, torch.Tensor):
        xh = out
    elif isinstance(out, (list, tuple)):
        xh = next((t for t in out if isinstance(t, torch.Tensor) and t.dim()==4), None)
        if xh is None:
            raise TypeError("No 4D tensor found in tuple/list output.")
    elif isinstance(out, dict):
        for k in ("x_hat", "recon", "out", "reconstruction"):
            if k in out and isinstance(out[k], torch.Tensor) and out[k].dim()==4:
                xh = out[k]; break
        else:
            xh = next((v for v in out.values() if isinstance(v, torch.Tensor) and v.dim()==4), None)
            if xh is None:
                raise TypeError("No 4D tensor found in dict output.")
    else:
        raise TypeError(f"Unsupported output type: {type(out)}")

    if from_logits:
        xh = torch.sigmoid(xh)
    return xh




@torch.no_grad()
def check_endpoint_agreement(vae, loader, from_logits=False):
    vae.eval()
    x,_ = next(iter(loader)); x = x.to(device)[:8]
    mu, lv = vae.encode(x)
    x_det = vae.decode(mu)
    if from_logits: x_det = torch.sigmoid(x_det)
    x_det = x_det.clamp(0,1)
    out = vae(x)                         # may sample z = mu + sigma*eps
    x_samp = extract_recon(out, from_logits=from_logits).clamp(0,1)
    diff = (x_det - x_samp).abs().mean().item()
    print(f"deterministic (decode(mu)) vs sampled forward | MAE: {diff:.5f}")


check_endpoint_agreement(conv_vae_model, val_loader)

@torch.no_grad()
def kl_and_latent_stats(vae, loader, n_batches=4):
    vae.eval()
    tot_kld, tot = 0.0, 0
    mus, stds = [], []
    for i, (x,_) in enumerate(loader):
        if i >= n_batches: break
        x = x.to(device)
        mu, lv = vae.encode(x)
        std = (0.5*lv).exp()
        # conv-latent: average across C,H,W
        kld = (-0.5 * (1 + lv - mu.pow(2) - lv.exp())).mean(dim=[1,2,3])  # per-sample
        tot_kld += kld.sum().item(); tot += kld.numel()
        mus.append(mu.detach().mean().item()); stds.append(std.detach().mean().item())
    print(f"avg KL per sample: {tot_kld/tot:.4f} | mean(μ): {sum(mus)/len(mus):.3f} | mean(σ): {sum(stds)/len(stds):.3f}")

kl_and_latent_stats(conv_vae_model, val_loader)

@torch.no_grad()
def psnr_ssim(model, loader, is_vae=False, from_logits=False, n_batches=10):
    model.eval()
    import math
    try:
        from skimage.metrics import structural_similarity as ssim
        has_ssim = True
    except Exception:
        has_ssim = False

    def _psnr(a,b):
        mse = F.mse_loss(a, b)
        return 10.0 * math.log10(1.0 / (mse.item() + 1e-12))

    p_list, s_list = [], []
    for i, (x, _) in enumerate(loader):
        if i >= n_batches: break
        x = x.to(device)
        out = model(x)               # VAE likely returns (x_hat, mu, logvar)
        xh  = extract_recon(out, from_logits=from_logits).clamp(0,1)

        p_list.append(_psnr(xh, x))
        if has_ssim:
            xs = x.permute(0,2,3,1).cpu().numpy()
            rs = xh.permute(0,2,3,1).cpu().numpy()
            for j in range(xs.shape[0]):
                s_list.append(ssim(xs[j], rs[j], channel_axis=2, data_range=1.0))

    msg = f"PSNR: {sum(p_list)/len(p_list):.2f} dB"
    if s_list: msg += f" | SSIM: {sum(s_list)/len(s_list):.3f}"
    print(msg)

psnr_ssim(conv_vae_model, val_loader, is_vae=True, from_logits=False)

@torch.no_grad()
def laplacian_energy(img):  # img: BxCxHxW in [0,1]
    k = torch.tensor([[0,1,0],[1,-4,1],[0,1,0]], dtype=img.dtype, device=img.device).view(1,1,3,3)
    if img.size(1) > 1:  # convert to grayscale for simplicity
        img = img.mean(dim=1, keepdim=True)
    return torch.nn.functional.conv2d(img, k, padding=1).pow(2).mean().item()

@torch.no_grad()
def compare_laplacian(model, loader, is_vae=False):
    x,_ = next(iter(loader)); x = x.to(device)[:8]
    xh = (model(x)[0] if is_vae else model(x)).clamp(0,1)
    print("Laplacian energy | input:", laplacian_energy(x),
          " recon:", laplacian_energy(xh))

compare_laplacian(conv_vae_model, val_loader, is_vae=True)

In [ ]:
xb,_ = next(iter(train_loader)); xb = xb[:4]
print("range:", xb.min().item(), xb.max().item())
# ~0..1 => no Normalize used; around -2..+2 (or similar) => Normalize used
# 3 RGB channels + sigmoid head
print("VAE out shape:", conv_vae_model(xb[:2].to(device))[0].shape)  # expect (B,3,H,W)
def recon_loss(x_hat, x):
    # better color than pure MSE
    return 0.8*torch.nn.functional.l1_loss(x_hat, x) + 0.2*torch.nn.functional.mse_loss(x_hat, x)

@torch.no_grad()
def channel_stats(model, loader, device, n=64):
    model.eval()
    xs, rs = [], []
    for i, (x, _) in enumerate(loader):
        x = x.to(device); xh = model(x)[0]  # VAE: (recon, mu, logvar)
        xs.append(x[:n]); rs.append(xh[:n])
        if sum(t.size(0) for t in xs) >= n: break
    X  = torch.cat(xs)[:n]; R = torch.cat(rs)[:n]
    for name, T in [("input", X), ("recon", R)]:
        m = T.mean(dim=[0,2,3]).cpu().numpy()
        s = T.std(dim=[0,2,3]).cpu().numpy()
        print(f"{name} per-channel mean/std:", list(zip(m.round(3), s.round(3))))
# call it:
channel_stats(conv_vae_model, val_loader, device)



# Class Takeaways — AE vs VAE vs U-Net (with & without skips)

**Big idea:** Same encoder–decoder *bones*; different *heads & wiring* yield different superpowers.



### Reconstruction (AE ↔ VAE)
- **AE:** Deterministic latent `z`; minimizes recon loss (MSE/BCE). Fast, but latent space can be lumpy.
- **VAE:** Learns a **distribution** `q(z|x)`; uses **reparameterization** and **KL(q‖p)** to a prior → smoother, more structured latent space.
- **Why you care:** VAEs generalize better and support **latent walks** (interpolations that stay “on-manifold”).
- **What to show:** AE vs VAE recon row + a **latent walk**; VAE transitions are smoother and semantically coherent.



### Segmentation (No-Skip ↔ U-Net)
- **No-Skip:** Coarse blobs; loses detail when decoding from compressed features.
- **U-Net:** **Skip concatenations** pipe high-res **edges & textures** straight to the decoder → sharper boundaries.
- **Why you care:** Skips buy you **crisp masks** without a massive model.
- **What to show:** Same images, columns: Input → AE → VAE → GT → **U-Net pred** → **No-Skip pred**.
  - Expect U-Net to beat No-Skip on **IoU** (especially ears/whiskers/tails).



### Metrics that matter
- **IoU (Jaccard):** `TP / (TP + FP + FN)` — go-to overlap score.
- **IoU vs Threshold curve:** Single-peaked; pick the **best threshold** (often 0.5–0.7). Don’t lock into 0.5 by faith alone.
- **Dice + BCE (loss):** Dice fights class imbalance; BCE stabilizes logits. Great combo for single-class seg.



### Demo anatomy (what to highlight)
1. **VAE latent walk:** Smooth morphs = “latent space learned structure.”
2. **U-Net vs No-Skip:** Side-by-side masks — U-Net wins on boundaries.
3. **Training curves:** U-Net ramps slower early, then overtakes. No-Skip plateaus first.



### Common gotchas (aka why things look cursed)
- Masks must be **{0,1}** and resized with **nearest** neighbor (never bilinear).
- **Skip concat** must actually concatenate channels (not sum), with correct channel math.
- Small batch? Prefer **GroupNorm** over BatchNorm.
- Head bias: initialize to foreground prior (optional, helps early convergence).



### TL;DR for students
- **VAE**: adds KL + sampling → **structured latent** → better recon & interpolation.  
- **U-Net**: adds **skip-concat** → **sharper masks** and higher IoU than no-skip.  
- Same skeleton, different add-ons; pick the head/wiring for the job.


# Remember this
### VAEs make the latent space behave; U-Nets make the boundaries behave.

# Archive